In [1]:
# 第5章｜用 Python API 自動下單：每次開啟 notebook，先執行這一格（安裝套件、登入 FinLab、開啟資料快取）
%pip install -q finlab==2.0.21 ta-lib==0.8.1
import finlab
from finlab import data

finlab.login()
data.set_storage(data.FileStorage())

Note: you may need to restart the kernel to use updated packages.


已登入（使用快取憑證）。


# 用 Python API 自動下單

**對應影片**：第 5 章 單元 2、3「用 Python API 自動下單 Part 1、Part 2」

**和影片的差異**

| 影片（2018） | 新版教材 |
| --- | --- |
| 直接呼叫券商 API，自己計算買賣張數、逐筆下單 | 用 `finlab.online`：`Position` 算出目標張數，`OrderExecutor` 比對帳戶現有部位、自動產生買賣委託 |
| 只支援當時的單一券商 | 同一套程式碼支援永豐、富邦、元富、玉山富果等券商，換券商只要換帳戶物件 |

**這本 notebook 不會送出真實委託單。** 我們用一個「模擬帳戶」走完整個流程；最後一段說明換成真實券商帳戶的方法。

Part 1：從回測結果算出目標部位。Part 2：比對帳戶、產生委託單。

In [2]:
import datetime
import itertools

import pandas as pd
from finlab.backtest import sim
from finlab.online import Account, Action, Order, OrderCondition, OrderExecutor, OrderStatus, Position, Stock

## Part 1：從策略到目標部位

### 1. 策略

沿用第 1 章的營收動能選股：股價在 60 日均線之上、近 3 月營收平均大於近 12 月、成交量足夠，每月挑營收動能最強的 `N_STOCKS` 檔。
這個單元的重點是下單流程，策略可以換成上一個單元的 ML 策略或你自己的策略，後面的步驟都一樣。

In [3]:
N_STOCKS = 10
MA_WINDOW = 60
REVENUE_SHORT_WINDOW, REVENUE_LONG_WINDOW = 3, 12
LIQUIDITY_WINDOW = 20
MIN_AVG_VOLUME_SHARES = 500_000

close = data.get('price:收盤價')
revenue = data.get('monthly_revenue:當月營收')
volume = data.get('price:成交股數')

revenue_short, revenue_long = revenue.average(REVENUE_SHORT_WINDOW), revenue.average(REVENUE_LONG_WINDOW)
revenue_momentum = revenue_short / revenue_long
condition = (
    (close > close.average(MA_WINDOW))
    & (revenue_short > revenue_long)
    & (volume.average(LIQUIDITY_WINDOW) > MIN_AVG_VOLUME_SHARES)
)
position = (revenue_momentum * condition).is_largest(N_STOCKS)

report = sim(position, resample='ME', name='Revenue momentum')
{key: report.get_stats()[key] for key in ['cagr', 'max_drawdown', 'daily_sharpe']}

{'cagr': -0.004905645060160446,
 'max_drawdown': -0.6660005132647012,
 'daily_sharpe': 0.040443081954855534}

### 2. 最新一期的持股權重

回測的持股表最後一列，就是「如果今天照策略操作，應該持有哪些股票」。等權重分配：

In [4]:
latest_date = position.index[-1]
latest_holdings = position.loc[latest_date]
weights = latest_holdings[latest_holdings] / latest_holdings.sum()

categories = data.get('security_categories').set_index('stock_id')
print(f'資料日期：{latest_date:%Y-%m-%d}')
pd.DataFrame({'name': categories['name'].reindex(weights.index), 'weight': weights})

資料日期：2018-12-28


,name,weight
symbol,,
1515,力山,0.1
1730,花仙子,0.1
2404,漢唐,0.1
2501,國建,0.1
2505,國揚,0.1
2520,冠德,0.1
5457,宣德,0.1
5519,隆大,0.1
8038,長園科,0.1


### 3. 權重 → 張數：`Position.from_weight`

給定資金與股價，`Position.from_weight` 會算出每檔要買幾張。預設只買整張；`odd_lot=True` 允許零股，資金可以分配得更平均。

In [5]:
FUND = 1_000_000

latest_price = close.loc[latest_date, weights.index]

target = Position.from_weight(weights, fund=FUND, price=latest_price)
target_with_odd_lot = Position.from_weight(weights, fund=FUND, price=latest_price, odd_lot=True)


def lots_table(position: Position) -> pd.Series:
    return pd.Series({p['stock_id']: p['quantity'] for p in position.position}, name='lots')


pd.DataFrame({'board lots only': lots_table(target), 'with odd lots': lots_table(target_with_odd_lot)}).fillna(0)

,board lots only,with odd lots
1515,2.0,1.398
1730,2.0,1.557
2404,1.0,1.115
2501,5.0,5.012
2505,8.0,8.13
2520,5.0,4.89
5457,2.0,1.801
5519,8.0,8.065
8038,2.0,1.584
8436,0.0,0.193


## Part 2：產生委託單

### 4. 模擬帳戶

`OrderExecutor` 需要一個帳戶物件，才能查詢現有部位與報價、送出委託。每家券商的帳戶類別（例如 `SinopacAccount`）都實作同一組方法。

下面的 `PaperAccount` 也實作了這組方法，但委託只記錄在記憶體裡，**不會送到任何券商**。報價使用 FinLab 資料中的最新收盤價。

In [6]:
SHARES_PER_LOT = 1000


class PaperAccount(Account):
    """模擬帳戶：記錄委託、不連線券商，用來練習下單流程。"""

    def __init__(self, holdings: dict[str, float], prices: pd.Series, cash: int):
        self.holdings = holdings
        self.prices = prices
        self.cash = cash
        self.orders: dict[str, Order] = {}
        self._order_ids = itertools.count(1)

    def create_order(self, action, stock_id, quantity, price=None, odd_lot=False, market_order=False,
                     best_price_limit=False, order_cond=OrderCondition.CASH):
        order_id = f'PAPER{next(self._order_ids):04d}'
        self.orders[order_id] = Order(
            order_id=order_id, stock_id=stock_id, action=action, price=price, quantity=quantity,
            filled_quantity=0, status=OrderStatus.NEW, order_condition=order_cond,
            time=datetime.datetime.now(),
        )
        return order_id

    def update_order(self, order_id, price=None, quantity=None):
        order = self.orders[order_id]
        order.price = price if price is not None else order.price
        order.quantity = quantity if quantity is not None else order.quantity

    def cancel_order(self, order_id):
        self.orders[order_id].status = OrderStatus.CANCEL

    def get_orders(self):
        return self.orders

    def get_stocks(self, stock_ids):
        return {
            sid: Stock(stock_id=sid, open=price, high=price, low=price, close=price,
                       bid_price=price, bid_volume=0, ask_price=price, ask_volume=0)
            for sid in stock_ids if pd.notna(price := self.prices.get(sid))
        }

    def get_position(self):
        return Position(self.holdings)

    def get_total_balance(self):
        stock_value = sum(self.prices[sid] * lots * SHARES_PER_LOT for sid, lots in self.holdings.items())
        return self.cash + stock_value

    def get_cash(self):
        return self.cash

    def get_settlement(self):
        return 0

假設帳戶裡已經有兩檔股票：一檔是策略這期也要持有的，另一檔是策略不要的。

In [7]:
CASH = 500_000

kept_stock = weights.index[0]
unwanted_stock = '2330' if '2330' not in weights.index else '2317'
account = PaperAccount(
    holdings={kept_stock: 1, unwanted_stock: 1},
    prices=close.loc[latest_date],
    cash=CASH,
)
print('帳戶現有部位：', account.get_position())

帳戶現有部位： symbol stock_id  quantity order_condition
  1515     1515         1            CASH
  2330     2330         1            CASH


### 5. 比對部位、預覽委託單

`OrderExecutor(目標部位, 帳戶)` 會計算「目標 − 現有」的差額：
已經持有的股票只補差額、策略不要的股票賣掉、新的股票買進。

`create_orders(view_only=True)` 只印出委託內容，不會送出。**第一次接真實帳戶時，一定先用 `view_only=True` 檢查。**

In [8]:
executor = OrderExecutor(target, account=account)
preview = executor.create_orders(view_only=True)

BUY         2505       X 8.0        @ 12.3         CASH
BUY         5457       X 2.0        @ 55.5         CASH
BUY         1730       X 2.0        @ 64.2         CASH
BUY         2404       X 1.0        @ 89.7         CASH
BUY         1515       X 1.0        @ 71.5         CASH
BUY         8038       X 2.0        @ 63.1         CASH
BUY         2520       X 5.0        @ 20.45        CASH
BUY         5519       X 8.0        @ 12.4         CASH
SELL        2330       X 1.0        @ 225.5        CASH
BUY         2501       X 5.0        @ 19.95        CASH


### 6. 送出委託（模擬帳戶）

拿掉 `view_only`，委託就會交給帳戶物件。模擬帳戶只把它記下來：

In [9]:
executor.create_orders()

pd.DataFrame([
    {'order_id': o.order_id, 'stock_id': o.stock_id, 'action': Action(o.action).name,
     'quantity': o.quantity, 'price': o.price, 'status': OrderStatus(o.status).name}
    for o in account.get_orders().values()
])

BUY         2505       X 8.0        @ 12.3         CASH
BUY         5457       X 2.0        @ 55.5         CASH
BUY         1730       X 2.0        @ 64.2         CASH
BUY         2404       X 1.0        @ 89.7         CASH
BUY         1515       X 1.0        @ 71.5         CASH
BUY         8038       X 2.0        @ 63.1         CASH
BUY         2520       X 5.0        @ 20.45        CASH
BUY         5519       X 8.0        @ 12.4         CASH
SELL        2330       X 1.0        @ 225.5        CASH
BUY         2501       X 5.0        @ 19.95        CASH


,order_id,stock_id,action,quantity,price,status
0,PAPER0001,2505,BUY,8,12.30,NEW
1,PAPER0002,5457,BUY,2,55.50,NEW
2,PAPER0003,1730,BUY,2,64.20,NEW
3,PAPER0004,2404,BUY,1,89.70,NEW
4,PAPER0005,1515,BUY,1,71.50,NEW
5,PAPER0006,8038,BUY,2,63.10,NEW
6,PAPER0007,2520,BUY,5,20.45,NEW
7,PAPER0008,5519,BUY,8,12.40,NEW
8,PAPER0009,2330,SELL,1,225.50,NEW
9,PAPER0010,2501,BUY,5,19.95,NEW


注意：零股和整股是分開下單的，所以同一檔股票可能出現兩筆委託（例如 1 張 + 300 股）。

## 換成真實券商帳戶

1. 向券商申請 API 使用權限與憑證（永豐：Shioaji；富邦：Fubon Neo API）。
2. 安裝券商套件：永豐 `pip install shioaji`；富邦的 `fubon_neo` 請到富邦官網下載。
3. 把帳號資訊設成**環境變數**（不要寫在 notebook 裡，更不要上傳到 GitHub）：

| 券商 | 帳戶類別 | 需要的環境變數 |
| --- | --- | --- |
| 永豐 | `finlab.online.brokers.sinopac.SinopacAccount` | `SHIOAJI_API_KEY`、`SHIOAJI_SECRET_KEY`、`SHIOAJI_CERT_PERSON_ID`、`SHIOAJI_CERT_PATH`、`SHIOAJI_CERT_PASSWORD` |
| 富邦 | `finlab.online.brokers.fubon.FubonAccount` | `FUBON_NATIONAL_ID`、`FUBON_ACCOUNT`、`FUBON_ACCOUNT_PASS`、`FUBON_CERT_PATH`、`FUBON_CERT_PASS` |

4. 把 `PaperAccount(...)` 換成券商帳戶，其餘程式碼完全不變。下面這格預設不會執行，確認上面的步驟都完成、也先用 `view_only=True` 檢查過委託內容，再把 `USE_REAL_BROKER` 改成 `True`。

In [10]:
USE_REAL_BROKER = False

if USE_REAL_BROKER:
    from finlab.online.brokers.sinopac import SinopacAccount

    real_account = SinopacAccount()
    real_target = Position.from_weight(weights, fund=FUND, price=real_account.get_price(list(weights.index)))
    OrderExecutor(real_target, account=real_account).create_orders(view_only=True)

實際運作時，通常每天收盤後重新執行：更新資料 → 回測取得最新持股 → 比對帳戶 → 下一個交易日下單。
自動下單牽涉真實金錢，建議先用小資金跑一段時間，確認每天的委託都符合預期。